# Visualize Binary Classification Eval Dataset

Interactive browser: each cell loads one subcategory. Press **Enter** to see the next example, **Esc** to stop.

Reads from `dataset/` directory (one `.jsonl` per subcategory).

In [2]:
import json
import sys
from pathlib import Path

import chess
import chess.svg
from IPython.display import SVG, display, HTML, clear_output

DATA_DIR = Path('dataset')
TAXONOMY_PATH = Path('taxonomy.json')

with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

def load_subcategory(name: str) -> list[dict]:
    """Load rows from a subcategory JSONL file."""
    path = DATA_DIR / f'{name}.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.open()]


def render_example(row: dict, idx: int, total: int, size: int = 350) -> str:
    """Render a single example as HTML with board + move info."""
    board = chess.Board(row['fen'])
    
    # Highlight the move square(s)
    from_sq = chess.parse_square(row['move_uci'][:2])
    to_sq = chess.parse_square(row['move_uci'][2:4])
    
    label_color = '#00aa00' if row['label'] == 'legal' else '#cc0000'
    label_text = row['label'].upper()
    
    arrows = [chess.svg.Arrow(from_sq, to_sq, color=label_color)]
    svg = chess.svg.board(board, arrows=arrows, size=size)
    
    turn = 'White' if board.turn else 'Black'
    in_check = ' (in check)' if board.is_check() else ''
    
    return f"""
    <div style='display:flex; gap:24px; align-items:flex-start; margin:8px 0'>
        <div>{svg}</div>
        <div style='font-family:monospace; font-size:14px; line-height:1.8'>
            <div style='font-size:18px; font-weight:bold; color:{label_color}'>
                {label_text} — {row['subcategory']}
            </div>
            <div><b>Example:</b> {idx+1} / {total}</div>
            <div><b>Move:</b> {row['move_san']} ({row['move_uci']})</div>
            <div><b>Turn:</b> {turn}{in_check}</div>
            <div><b>Category:</b> {row['category']}</div>
            <div><b>Phase:</b> {row['phase']}</div>
            <div style='margin-top:8px'><b>FEN:</b></div>
            <div style='font-size:11px; word-break:break-all'>{row['fen']}</div>
        </div>
    </div>
    """


def browse(subcategory: str):
    """Interactive browser. Enter = next, Esc/q = stop."""
    rows = load_subcategory(subcategory)
    if not rows:
        print(f'No data for {subcategory} (file not found in {DATA_DIR}/)')
        return
    
    print(f'Loaded {len(rows)} examples for {subcategory}. Press Enter for next, type q to stop.')
    
    for i, row in enumerate(rows):
        clear_output(wait=True)
        html = render_example(row, i, len(rows))
        display(HTML(html))
        try:
            resp = input(f'[{i+1}/{len(rows)}] Enter=next, q=stop: ')
            if resp.strip().lower() in ('q', 'quit', 'exit'):
                break
        except (KeyboardInterrupt, EOFError):
            break
    
    print(f'Stopped at example {i+1}/{len(rows)}')


# Show available subcategories
available = sorted(p.stem for p in DATA_DIR.glob('*.jsonl'))
print(f'{len(available)} subcategories available: {available}')

36 subcategories available: ['backward_pawn', 'blocked_sliding', 'castling_in_check', 'castling_no_rights', 'castling_path_occupied', 'castling_through_attacked', 'ep_fake_diagonal', 'ep_pinned', 'ep_wrong_pawn', 'friendly_fire', 'king_to_attacked', 'legal_block_check', 'legal_capture', 'legal_capture_checker', 'legal_castling', 'legal_check', 'legal_en_passant', 'legal_king_escape', 'legal_move', 'legal_promotion', 'non_evasion_in_check', 'non_king_double_check', 'pawn_capture_friendly', 'pawn_diagonal_to_empty', 'pawn_double_push_blocked', 'pawn_double_wrong_rank', 'pawn_push_onto_piece', 'pin_breaking', 'promo_capture_empty', 'promo_push_blocked', 'wrong_ep', 'wrong_geometry_bishop', 'wrong_geometry_king', 'wrong_geometry_knight', 'wrong_geometry_queen', 'wrong_geometry_rook']


---
## Overview: First Example per Subcategory

In [3]:
# Show first example from each subcategory (grouped by category)
DESCRIPTIONS = {
    "legal_move": "Normal piece move",
    "legal_capture": "Captures an enemy piece",
    "legal_castling": "Legal castling",
    "legal_en_passant": "Legal en passant",
    "legal_promotion": "Pawn promotes",
    "legal_check": "Move delivers check",
    "legal_king_escape": "King moves out of check",
    "legal_capture_checker": "Non-king captures the checking piece",
    "legal_block_check": "Non-king interposes on the check ray",
    "non_evasion_in_check": "Non-king move that doesn't address check",
    "non_king_double_check": "Non-king move in double check",
    "king_to_attacked": "King moves to attacked square",
    "castling_in_check": "Castling while in check",
    "castling_through_attacked": "Castling through attacked square",
    "castling_path_occupied": "Pieces between king and rook",
    "castling_no_rights": "King or rook has already moved",
    "wrong_geometry_king": "King moves >1 square (non-castling)",
    "pin_breaking": "Pinned piece moves off pin ray",
    "backward_pawn": "Pawn moves backward",
    "pawn_double_wrong_rank": "Double push from non-starting rank",
    "pawn_double_push_blocked": "Double push with blocked intermediate",
    "pawn_push_onto_piece": "Pawn pushes into occupied square",
    "pawn_diagonal_to_empty": "Pawn diagonal to empty (no capture)",
    "pawn_capture_friendly": "Pawn captures own piece",
    "ep_fake_diagonal": "Diagonal in EP position, no adjacent enemy pawn",
    "ep_wrong_pawn": "Adjacent pawn didn't just double-push",
    "wrong_ep": "EP-like move but EP not available",
    "ep_pinned": "Correct EP but reveals lateral check on king",
    "promo_push_blocked": "Promotion push onto occupied square",
    "promo_capture_empty": "Promotion capture to empty square",
    "friendly_fire": "Piece captures own piece",
    "blocked_sliding": "Sliding piece moves through blocker",
    "wrong_geometry_knight": "Knight moves like bishop",
    "wrong_geometry_bishop": "Bishop moves like rook",
    "wrong_geometry_rook": "Rook moves like bishop",
    "wrong_geometry_queen": "Queen moves like knight",
}

SECTION_ORDER = [
    ("Legal", "legal", [
        "legal_move", "legal_capture", "legal_castling", "legal_en_passant",
        "legal_promotion", "legal_check", "legal_king_escape",
        "legal_capture_checker", "legal_block_check",
    ]),
    ("Illegal — Check Evasion", "check_evasion", [
        "non_evasion_in_check", "non_king_double_check",
    ]),
    ("Illegal — King", "king", [
        "king_to_attacked", "castling_in_check", "castling_through_attacked",
        "castling_path_occupied", "castling_no_rights", "wrong_geometry_king",
    ]),
    ("Illegal — Pin", "pin", ["pin_breaking"]),
    ("Illegal — Pawn", "pawn", [
        "backward_pawn", "pawn_double_wrong_rank", "pawn_double_push_blocked",
        "pawn_push_onto_piece", "pawn_diagonal_to_empty", "pawn_capture_friendly",
    ]),
    ("Illegal — En Passant", "en_passant", [
        "ep_fake_diagonal", "ep_wrong_pawn", "wrong_ep", "ep_pinned",
    ]),
    ("Illegal — Promotion", "promotion", [
        "promo_push_blocked", "promo_capture_empty",
    ]),
    ("Illegal — Piece Movement", "piece_movement", [
        "friendly_fire", "blocked_sliding", "wrong_geometry_knight",
        "wrong_geometry_bishop", "wrong_geometry_rook", "wrong_geometry_queen",
    ]),
]

html_parts = []
for section_title, category, subcats in SECTION_ORDER:
    html_parts.append(f"<h2>{section_title}</h2>")
    for sub in subcats:
        rows = load_subcategory(sub)
        desc = DESCRIPTIONS.get(sub, "")
        if not rows:
            html_parts.append(
                f"<div style='margin:8px 0; padding:8px; background:#fff3cd; border-radius:4px'>"
                f"<b>{sub}</b> — {desc} — <i>no data yet</i></div>"
            )
            continue
        row = rows[0]
        count = len(rows)
        html_parts.append(render_example(row, 0, count, size=280))
        html_parts.append(
            f"<div style='margin:-8px 0 16px 0; font-size:12px; color:#666'>"
            f"<b>{sub}</b> — {desc} — {count} examples</div>"
        )

display(HTML("".join(html_parts)))

---
## Legal

In [11]:
browse('legal_move')

Stopped at example 6/2000


In [12]:
browse('legal_capture')

Stopped at example 5/2000


In [13]:
browse('legal_castling')

Stopped at example 8/2000


In [14]:
browse('legal_en_passant')

Stopped at example 4/2000


In [15]:
browse('legal_promotion')

Stopped at example 18/2000


In [16]:
browse('legal_check')

Stopped at example 7/2000


In [17]:
browse('legal_king_escape')

Stopped at example 5/2000


In [18]:
browse('legal_capture_checker')

Stopped at example 5/2000


In [19]:
browse('legal_block_check')

Stopped at example 1/2000


---
## Illegal — Check Evasion

In [20]:
browse('non_evasion_in_check')

Stopped at example 4/2000


In [21]:
browse('non_king_double_check')

Stopped at example 4/1404


---
## Illegal — King

In [22]:
browse('king_to_attacked')

Stopped at example 5/2000


In [23]:
browse('castling_in_check')

Stopped at example 4/2000


In [24]:
browse('castling_through_attacked')

Stopped at example 1/2000


In [25]:
browse('castling_path_occupied')

Stopped at example 1/2000


In [26]:
browse('castling_no_rights')

No data for castling_no_rights (file not found in dataset/)


In [27]:
browse('wrong_geometry_king')

Stopped at example 1/2000


---
## Illegal — Pin

In [28]:
browse('pin_breaking')

Stopped at example 3/2000


---
## Illegal — Pawn

In [29]:
browse('backward_pawn')

Stopped at example 1/2000


In [ ]:
browse('pawn_double_wrong_rank')

In [ ]:
browse('pawn_double_push_blocked')

In [ ]:
browse('pawn_push_onto_piece')

In [ ]:
browse('pawn_diagonal_to_empty')

In [ ]:
browse('pawn_capture_friendly')

---
## Illegal — En Passant

In [ ]:
browse('ep_fake_diagonal')

Stopped at example 19/2000


In [ ]:
browse('ep_wrong_pawn')

Stopped at example 8/2000


In [ ]:
browse('wrong_ep')

In [ ]:
browse('ep_pinned')

---
## Illegal — Promotion

In [ ]:
browse('promo_push_blocked')

In [ ]:
browse('promo_capture_empty')

---
## Illegal — Piece Movement

In [ ]:
browse('friendly_fire')

Stopped at example 6/2000


In [ ]:
browse('blocked_sliding')

Stopped at example 17/2000


In [1]:
browse('wrong_geometry_knight')

NameError: name 'browse' is not defined

In [ ]:
browse('wrong_geometry_bishop')

In [ ]:
browse('wrong_geometry_rook')

In [ ]:
browse('wrong_geometry_queen')